<a href="https://colab.research.google.com/github/cedricnagata/skin_lesion_classifier/blob/main/main_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

# Set up Google Drive paths
DRIVE_PATH = '/content/drive/My Drive/Software Projects/skin_lesion_classifier/'

DATA_PATH = os.path.join(DRIVE_PATH, 'data')
METADATA_PATH = os.path.join(os.path.join(DATA_PATH, 'metadata'), 'balanced_metadata.csv')
IMAGES_PATH = os.path.join(os.path.join(DATA_PATH, 'image_folders'), 'images')

SCRIPTS_PATH = os.path.join(DRIVE_PATH, 'scripts')
MODELS_PATH = os.path.join(DRIVE_PATH, 'models')
RESULTS_PATH = os.path.join(DRIVE_PATH, 'results')

ZIP_PATH = os.path.join(DRIVE_PATH, 'h5_data.zip')

In [4]:
import shutil

# Copy the h5 zip from Google Drive to the local environment
shutil.copy2(ZIP_PATH, '/content/')
print("zip copied successfully.")

zip copied successfully.


In [5]:
import zipfile

print("unzipping...")

def unzip_file(zip_file_path, extract_dir):
  with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

unzip_file('/content/h5_data.zip', '/content/')

print("unzipped successfully.")

unzipping...
unzipped successfully.


In [19]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import sys
import h5py
import numpy as np
from PIL import Image

print(tf.__version__)

# Add the scripts path to the system path
sys.path.append(SCRIPTS_PATH)

# Import the necessary functions from the scripts
from preprocess_data import *
from build_model import *
from train_model import *
from evaluate_model import *

# matplotlib figures appear inline in the notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Auto reload python modules
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

2.17.0
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Initialize the TPU
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
tf.config.experimental_connect_to_cluster(resolver)
tf.tpu.experimental.initialize_tpu_system(resolver)
strategy = tf.distribute.TPUStrategy(resolver)

print("All devices: ", tf.config.list_logical_devices('TPU'))

All devices:  [LogicalDevice(name='/device:TPU:0', device_type='TPU'), LogicalDevice(name='/device:TPU:1', device_type='TPU'), LogicalDevice(name='/device:TPU:2', device_type='TPU'), LogicalDevice(name='/device:TPU:3', device_type='TPU'), LogicalDevice(name='/device:TPU:4', device_type='TPU'), LogicalDevice(name='/device:TPU:5', device_type='TPU'), LogicalDevice(name='/device:TPU:6', device_type='TPU'), LogicalDevice(name='/device:TPU:7', device_type='TPU')]


In [20]:
import h5py
import tensorflow as tf

# Paths to HDF5 files
train_file = '/content/h5_data/train_preprocessed_data.h5'
val_file = '/content/h5_data/val_preprocessed_data.h5'
test_file = '/content/h5_data/test_preprocessed_data.h5'

# Dataset parameters
batch_size = 512

"""
# Load preprocessed data into memory
train_images, train_diagnoses, train_benign_malignants = load_h5_data(train_file)
val_images, val_diagnoses, val_benign_malignants = load_h5_data(val_file)
test_images, test_diagnoses, test_benign_malignants = load_h5_data(test_file)

# Create TensorFlow datasets
train_ds = get_dataset(train_images, train_diagnoses, train_benign_malignants, batch_size)
val_ds = get_dataset(val_images, val_diagnoses, val_benign_malignants, batch_size)
test_ds = get_dataset(test_images, test_diagnoses, test_benign_malignants, batch_size)
"""

# Dataset parameters
img_shape = (224, 224, 3)  # Update based on your image shape
num_classes = 3  # Update based on your number of classes

# Create TensorFlow datasets with data augmentation
train_ds = get_dataset(train_file, batch_size, img_shape, num_classes)
val_ds = get_dataset(val_file, batch_size, img_shape, num_classes)
test_ds = get_dataset(test_file, batch_size, img_shape, num_classes)

print("Datasets created successfully.")


Datasets created successfully.


In [22]:
# Calculate class weights from metadata
metadata = pd.read_csv(METADATA_PATH)
diagnosis_class_weights = calculate_class_weights(metadata['diagnosis'])
benign_malignant_class_weights = calculate_class_weights(metadata['benign_malignant'])

# Build and compile the ResNet50 model
fine_tune = False
num_unfreeze_layers = 5
model = build_resnet50_model(
    fine_tune=fine_tune,
    num_unfreeze_layers=num_unfreeze_layers,
    diagnosis_weights=list(diagnosis_class_weights.values()),
    benign_malignant_weights=list(benign_malignant_class_weights.values())
)

# Summarize and visualize the model
summarize_and_visualize_model(model)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7             │ (None, 224, 224, 3)    │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ resnet50 (Functional)     │ (None, 7, 7, 2048)     │     23,587,712 │ input_layer_7[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ global_average_pooling2d… │ (None, 2048)           │              0 │ resnet50[0][0]         │
│ (GlobalAveragePooling2D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_2 (Dense)           │ (None, 1024)           │      2,098,176 │ global_average_poolin… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_2 (Dropout)       │ (None, 1024)           │              0 │ dense_2[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ diagnosis (Dense)         │ (None, 3)              │          3,075 │ dropout_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ benign_malignant (Dense)  │ (None, 1)              │          1,025 │ dropout_2[0][0]        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 25,689,988 (98.00 MB)

 Trainable params: 2,102,276 (8.02 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
#with strategy.scope():
# Train the model
history = train_model(model,
                      train_ds,
                      val_ds,
                      epochs=100,
                      models_path=MODELS_PATH,
                      model_name='skin_lesion_model.keras')

Epoch 1/100
      3/Unknown 303s 97s/step - benign_malignant_accuracy: 0.5055 - diagnosis_accuracy: 0.3263 - loss: 15.7560

In [ ]:
with strategy.scope():
  # Evaluate the model
  metrics = evaluate_model(model, test_ds, results_path=RESULTS_PATH, results_file='evaluation_metrics.txt')
  print(metrics)